# Expanded Practice Exercise: A/B Testing Website Versions
## Two-Sample t-Test with Assumption Checks, Effect Size, CI, More Practice & Simulation

**Goal:** Practice the full workflow of a two-sample t-test for independent groups while considering your audience (data literacy, subject knowledge) when interpreting and reporting results.

This is the **SKELETON** version — follow the markdown instructions, fill in the code cells yourself. Use the **solution.ipynb** to check your work or see alternate approaches after attempting.

### Scenario (from original exercise)
A company randomly sampled 100 site visitors. They showed the **old version** of their website to 50 visitors and the **new version** to the other 50. Time spent (minutes) on the site was recorded. The data is saved in `version_time.csv`.

You have already seen an overlaid histogram in the original prompt. Now we expand it into a complete, professional analysis suitable for a data analysis report.


## Flowchart of the Desired Analysis Outcome

```mermaid
flowchart TD
    Start[Start] --> Load[Load Data &amp; EDA<br/>Histograms, Summary Stats, Groupby]
    Load --> Hypotheses[Formulate Hypotheses<br/>H0: μ_new = μ_old<br/>Ha: μ_new ≠ μ_old | α = 0.05 two-sided]
    Hypotheses --> Assumptions{Check Assumptions<br/>1. Normality (Shapiro-Wilk / QQ-plot / Hist)<br/>2. Equal Variance (Levene's test)}
    Assumptions -->|Pass| TTest[Run Two-Sample t-test<br/>scipy.stats.ttest_ind<br/>+ Alternates: statsmodels, manual numpy]
    Assumptions -->|Fail or Borderline| NonParam[Consider Mann-Whitney U<br/>or data transform / CLT justification]
    TTest --> EffectSize[Compute Effect Size<br/>Cohen's d + Interpretation]
    TTest --> CI[Compute 95% CI for Mean Difference]
    EffectSize --> Interpret[Interpret Results<br/>p-value vs α<br/>Statistical + Practical Significance]
    CI --> Interpret
    NonParam --> Interpret
    Interpret --> Audience[Consider Audience for Reporting<br/>- Executives: High-level business impact<br/>- Data team: Full stats, assumptions, code<br/>- Non-technical: Simple language + viz]
    Audience --> Conclusion[Conclusion &amp; Recommendations<br/>Rollout decision? Next experiments?]
    Conclusion --> Sim[Simulation: Power Analysis<br/>Modify params → observe power / Type I error]
    Sim --> End[End: Practice + Document Insights]
```

**Note:** Mermaid flowchart renders in JupyterLab, VS Code, nbviewer, or paste at https://mermaid.live. It shows the logical flow of a complete, audience-aware A/B test analysis.


## 1. Imports and Data Loading

**Instructions:**
- Import the necessary libraries: `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.stats`
- Load the CSV into a DataFrame called `data`
- Separate into two Series: `old` and `new` (as in the original exercise)
- Print the first few rows and value counts to verify


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
# Optional for alternates
# import statsmodels.stats.weightstats as smw   # if available

# TODO: Load the data
data = pd.read_csv('version_time.csv')

# TODO: Separate groups
old = data.time_minutes[data.version == 'old']
new = data.time_minutes[data.version == 'new']

print('Data shape:', data.shape)
print('Version counts:')
print(data['version'].value_counts())
print('\nFirst 5 rows:')
print(data.head())


## 2. Exploratory Data Analysis (EDA) & Visualization

**Instructions:**
1. Compute and print descriptive statistics for each group separately (mean, std, min, max, count). Use `.describe()` or groupby.
2. Create an overlaid histogram (as in original) with transparency (`alpha=0.6` or `0.8`), labels, legend, title 'Time Spent on Website by Version'.
3. (Bonus) Add side-by-side boxplots or violin plots to compare distributions visually.
4. Briefly note from visuals: Does the new version appear to increase time spent? Any outliers or skewness?

**Audience tip:** For less data-literate stakeholders, histograms + simple mean comparison are safer than advanced plots. Always label clearly.


In [ ]:
# TODO: Summary statistics per group
print('=== OLD version ===')
print(old.describe())
print('\n=== NEW version ===')
print(new.describe())

# TODO: Overlaid histogram
plt.figure(figsize=(8,5))
plt.hist(old, alpha=0.7, label='Old Version', bins=15, color='steelblue', edgecolor='white')
plt.hist(new, alpha=0.7, label='New Version', bins=15, color='coral', edgecolor='white')
plt.xlabel('Time spent (minutes)')
plt.ylabel('Number of visitors')
plt.title('Distribution of Time Spent on Website by Version')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

# TODO (optional): Boxplots
# data.boxplot(column='time_minutes', by='version')
# plt.title('Boxplot Comparison')
# plt.suptitle('')
# plt.show()


## 3. Formulate Hypotheses and Choose Significance Level

**Instructions:**
- State the null and alternative hypotheses clearly (use LaTeX style in markdown if desired).
- Choose α (commonly 0.05). Justify two-sided vs one-sided for this business question.
- In a markdown cell below, write your hypotheses.

**Example format:**
- H₀: μ_new = μ_old (no difference in mean time spent)
- Hₐ: μ_new ≠ μ_old (there is a difference)

This is a two-sided test because the company wants to detect if the new version is better *or* worse.


In [ ]:
# No code needed here - write your answer in the markdown cell above or a new one.
# Later you will use the p-value to decide whether to reject H0.
print('Hypotheses stated in markdown cell. α = 0.05 (two-sided)')


## 4. Check Statistical Assumptions

**Why this matters:** The two-sample t-test assumes:
1. Independence (satisfied by random assignment in A/B test)
2. Normality of each group's data (or large n → CLT)
3. Homogeneity of variances (equal spread)

**Instructions:**
1. Create Q-Q plots for both groups (use `stats.probplot` + matplotlib).
2. Run Shapiro-Wilk normality test on `old` and on `new`. Print p-values.
3. Run Levene's test for equal variances. Print p-value.
4. In a markdown cell: Decide — are assumptions reasonably met? What would you do if normality failed (e.g. n=50 is borderline, CLT often saves us; or switch to Mann-Whitney U)?

**Audience consideration:** Technical supervisors will want to see these checks documented. Executives usually trust you did them.


In [ ]:
# TODO: Q-Q plots (two subplots)
fig, axes = plt.subplots(1, 2, figsize=(10,4))
stats.probplot(old, dist='norm', plot=axes[0])
axes[0].set_title('Q-Q Plot: Old Version')
stats.probplot(new, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: New Version')
plt.tight_layout()
plt.show()

# TODO: Shapiro-Wilk
shap_old = stats.shapiro(old)
shap_new = stats.shapiro(new)
print(f'Shapiro-Wilk old: statistic={shap_old.statistic:.4f}, p-value={shap_old.pvalue:.4f}')
print(f'Shapiro-Wilk new: statistic={shap_new.statistic:.4f}, p-value={shap_new.pvalue:.4f}')

# TODO: Levene's test
lev = stats.levene(old, new)
print(f"Levene's test: statistic={lev.statistic:.4f}, p-value={lev.pvalue:.4f}")

# TODO: Your decision in markdown cell below this one


## 5. Perform the Two-Sample t-Test

**Instructions:**
1. Use `stats.ttest_ind(old, new)` (or with `equal_var=True`).
2. Save `tstat` and `pval`.
3. Print them nicely formatted.
4. Decide significance: `significant = pval < 0.05`
5. Also compute and print the observed mean difference (`new.mean() - old.mean()`).

**Alternate approaches to try later (see solution):**
- `statsmodels.stats.weightstats.ttest_ind`
- Manual calculation using numpy means + pooled std + `stats.t.sf` for p-value


In [ ]:
# TODO: Run the t-test
tstat, pval = stats.ttest_ind(old, new)
print(f't-statistic = {tstat:.4f}')
print(f'p-value     = {pval:.6f}')

# TODO: Significance decision
alpha = 0.05
significant = pval < alpha
print(f'Significant at α={alpha}? {significant}')

# TODO: Observed difference
mean_diff = new.mean() - old.mean()
print(f'Observed mean difference (new - old) = {mean_diff:.3f} minutes')


## 6. Effect Size and Confidence Interval

**Instructions:**
1. Calculate Cohen's d (pooled standard deviation version for independent samples).
   Formula: d = (mean_new - mean_old) / sqrt( ((n_old-1)*var_old + (n_new-1)*var_new) / (n_old + n_new - 2) )
2. Interpret: |d| ≈ 0.2 small, 0.5 medium, 0.8 large.
3. Compute 95% CI for the mean difference using `stats.t.interval` or manual (mean_diff ± t_crit * SE).
   SE = pooled_std / sqrt(1/n_old + 1/n_new)  or use `stats.ttest_ind` return if extended.

Print all values. In markdown below: Is the effect practically meaningful? How would different audiences react to d=0.63 vs just p-value?


In [ ]:
# TODO: Cohen's d
n_old, n_new = len(old), len(new)
var_old, var_new = old.var(ddof=1), new.var(ddof=1)   # sample variance
pooled_var = ((n_old-1)*var_old + (n_new-1)*var_new) / (n_old + n_new - 2)
pooled_std = np.sqrt(pooled_var)
cohens_d = mean_diff / pooled_std
print(f"Cohen's d = {cohens_d:.3f}  (medium effect if ~0.5)")

# TODO: 95% CI for mean difference
se_diff = pooled_std * np.sqrt(1/n_old + 1/n_new)
t_crit = stats.t.ppf(1 - alpha/2, df=n_old + n_new - 2)
ci_lower = mean_diff - t_crit * se_diff
ci_upper = mean_diff + t_crit * se_diff
print(f'95% CI for mean difference: [{ci_lower:.3f}, {ci_upper:.3f}] minutes')


## 7. More Practice Exercises

Complete these in new code cells or on paper. Check your answers against the solution notebook.

**Practice 1:** Re-run the significance decision with α = 0.01 and α = 0.10. Does your conclusion change?

**Practice 2:** Perform a one-sided test (`alternative='greater'` in ttest_ind). Interpret in context: "Is there evidence the new version increases time spent?"

**Practice 3:** Compute a 90% CI instead of 95%. How does the width change?

**Practice 4:** Run the non-parametric alternative `stats.mannwhitneyu(old, new, alternative='two-sided')`. Compare p-value and conclusion to the t-test.

**Practice 5 (Advanced):** Implement a bootstrap CI for the mean difference (resample each group with replacement 5000+ times, compute diff of means each time, take 2.5th and 97.5th percentiles). Compare to parametric CI.


## 8. Simulation Section: Statistical Power & Sensitivity Analysis

**Goal:** Understand how results change when you modify true effect size, sample size, or variability. This is crucial for experimental design (how many visitors do we need to detect a meaningful lift?).

**Instructions:**
1. Modify the parameters in the code cell below (especially `true_mean_new`, `n_per_group`, `sigma`, `n_simulations`).
2. Run the cell multiple times with different values.
3. Observe:
   - When true_mean_new == true_mean_old → estimated Type I error rate should be close to α (false positives)
   - When true diff exists → estimated Power = proportion of simulations where p < α
   - Larger n or larger true diff → higher power
   - Larger sigma (noise) → lower power

Prints and a p-value histogram will help you see the distribution of results across 'parallel universes' of the experiment.

**Try these scenarios:**
- true_mean_new = 23.53 (no effect) → should get ~5% significant
- true_mean_new = 26.88 (observed) with n=50 → high power?
- true_mean_new = 25.0 (smaller lift) with n=30 → lower power?
- Increase n_per_group to 200


In [ ]:
np.random.seed(42)   # for reproducibility

# === MODIFIABLE PARAMETERS - CHANGE THESE AND RE-RUN ===
true_mean_old = 23.53
true_mean_new = 26.88   # <-- change to 23.53 for null, or 25.0 for smaller effect
sigma = 5.3             # <-- noise level; try 8.0 for more variability
n_per_group = 50        # <-- sample size per version; try 30 or 200
n_simulations = 1000    # <-- number of Monte Carlo runs; 500-5000 typical
alpha = 0.05

# === Simulation loop (skeleton - complete the TODOs) ===
significant_count = 0
pvals = []

for i in range(n_simulations):
    old_sim = np.random.normal(true_mean_old, sigma, n_per_group)
    new_sim = np.random.normal(true_mean_new, sigma, n_per_group)
    # TODO: run t-test on simulated data
    _, p = stats.ttest_ind(old_sim, new_sim)
    pvals.append(p)
    if p < alpha:
        significant_count += 1

power_estimate = significant_count / n_simulations
label = 'Power (true effect exists)' if abs(true_mean_new - true_mean_old) > 0.1 else 'Type I Error Rate (no true effect)'
print(f'{label}: {power_estimate:.3f}  (based on {n_simulations} simulations)')
print(f'Mean p-value across sims: {np.mean(pvals):.4f}')

# TODO: Plot p-value distribution
plt.figure(figsize=(8,4))
plt.hist(pvals, bins=30, alpha=0.75, color='purple', edgecolor='white')
plt.axvline(alpha, color='red', linestyle='--', linewidth=2, label=f'α = {alpha}')
plt.xlabel('p-value')
plt.ylabel('Frequency (out of simulations)')
plt.title(f'Distribution of p-values from {n_simulations} Simulated Experiments')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

print('\nInterpretation tip: Under the null (no effect), p-values should be roughly uniform [0,1]. A peak near 0 indicates good power to detect the effect you simulated.')


## 9. Conclusion & Audience-Aware Reporting (Your Turn)

After completing the analysis:

1. Write a short **Conclusion** section suitable for a data analysis report (see the provided `paper-structure.pdf` guidance).
2. Consider the **audience** (from the PDFs `What to Consider When Considering the Audience.pdf` and `Audience and Situation Analysis.pdf`):
   - **Executives / Primary client**: High-level takeaway + business recommendation. Avoid jargon. Use the mean difference and a simple viz.
   - **Technical supervisor**: Full assumption checks, exact methods, effect size, CI, any limitations (e.g. only time metric, short test duration).
   - **Mixed or non-technical**: Explain in plain language what the p-value and d mean ("only ~0.2% chance of seeing this large a difference if new version had no real effect").

**Prompt for your write-up:**
> The new website version was associated with visitors spending approximately X minutes more on average (95% CI: [L, U]). This difference was statistically significant (p = Y) with a medium effect size (Cohen's d = Z). Assumptions of normality and equal variance were met. We recommend [rollout / further testing].

Add 1-2 sentences tailored to each audience type.

Save your insights — this notebook + your written conclusion can become part of your data science portfolio!
